In [4]:
import pandas as pd
import json

print("Loading data...")
# Load the raw files directly from Colab's root folder
ledger = pd.read_csv('ledger.csv')
gateway = pd.read_csv('gateway.csv')

# Clean transaction IDs to ensure perfect matching
ledger['transaction_id'] = ledger['transaction_id'].astype(str).str.strip()
gateway['transaction_id'] = gateway['transaction_id'].astype(str).str.strip()

print(f"Ledger rows: {len(ledger)} | Gateway rows: {len(gateway)}")
print("--- Validation Checks ---")
print(f"Ledger duplicate IDs: {ledger.duplicated(subset=['transaction_id']).sum()}")
print(f"Gateway duplicate IDs: {gateway.duplicated(subset=['transaction_id']).sum()}")
print(f"Ledger null amounts: {ledger['amount_usd'].isnull().sum()}")
print(f"Gateway null amounts: {gateway['amount_usd'].isnull().sum()}")

# Perform the Reconciliation
recon = pd.merge(ledger, gateway, on='transaction_id', how='outer', suffixes=('_ledg', '_gw'), indicator=True)

missing_in_gateway = recon[recon['_merge'] == 'left_only'].copy()
missing_in_ledger = recon[recon['_merge'] == 'right_only'].copy()

both = recon[recon['_merge'] == 'both'].copy()
amount_mismatches = both[both['amount_usd_ledg'] != both['amount_usd_gw']].copy()
status_mismatches = both[both['status_ledg'] != both['status_gw']].copy()

print(f"Missing in Gateway: {len(missing_in_gateway)}")
print(f"Missing in Ledger: {len(missing_in_ledger)}")
print(f"Amount Mismatches: {len(amount_mismatches)}")
print(f"Status Mismatches: {len(status_mismatches)}")

# Save Output CSVs directly to Colab
missing_in_gateway.to_csv('missing_in_gateway.csv', index=False)
missing_in_ledger.to_csv('missing_in_ledger.csv', index=False)
amount_mismatches.to_csv('amount_mismatches.csv', index=False)
status_mismatches.to_csv('status_mismatches.csv', index=False)

recon_report = recon.copy()
recon_report['issue_type'] = 'Matched'
recon_report.loc[recon_report['_merge'] == 'left_only', 'issue_type'] = 'Missing in Gateway'
recon_report.loc[recon_report['_merge'] == 'right_only', 'issue_type'] = 'Missing in Ledger'
recon_report.loc[(recon_report['_merge'] == 'both') & (recon_report['amount_usd_ledg'] != recon_report['amount_usd_gw']), 'issue_type'] = 'Amount Mismatch'
recon_report.loc[(recon_report['_merge'] == 'both') & (recon_report['status_ledg'] != recon_report['status_gw']), 'issue_type'] = 'Status Mismatch'

recon_report.to_csv('reconciliation_report.csv', index=False)

# Generate Summary Metrics JSON
amount_at_risk = float(missing_in_ledger['amount_usd_gw'].sum() + missing_in_gateway['amount_usd_ledg'].sum() + abs(amount_mismatches['amount_usd_ledg'] - amount_mismatches['amount_usd_gw']).sum())

metrics = {
  "total_ledger_rows": int(len(ledger)),
  "total_gateway_rows": int(len(gateway)),
  "missing_in_gateway_count": int(len(missing_in_gateway)),
  "missing_in_ledger_count": int(len(missing_in_ledger)),
  "amount_mismatch_count": int(len(amount_mismatches)),
  "status_mismatch_count": int(len(status_mismatches)),
  "reconciliation_issue_count": int(len(missing_in_gateway) + len(missing_in_ledger) + len(amount_mismatches) + len(status_mismatches)),
  "ledger_total_amount": float(ledger['amount_usd'].sum()),
  "gateway_total_amount": float(gateway['amount_usd'].sum()),
  "amount_at_risk": round(amount_at_risk, 2)
}

with open('summary_metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)

print("All tasks complete! Files are ready to download.")

Loading data...
Ledger rows: 10 | Gateway rows: 9
--- Validation Checks ---
Ledger duplicate IDs: 0
Gateway duplicate IDs: 0
Ledger null amounts: 0
Gateway null amounts: 0
Missing in Gateway: 2
Missing in Ledger: 1
Amount Mismatches: 2
Status Mismatches: 1
All tasks complete! Files are ready to download.
